# Membuat sparksession

In [3]:
import spark_session
import json
from pyspark.sql.functions import sum as spark_sum, col, when, count

In [4]:
app = spark_session.SparkApp("Tugas7")
spark = app.getSession()

Memulai SparkSession untuk Tugas7


26/09/30 08:48:31 WARN Utils: Your hostname, asfadani resolves to a loopback address: 127.0.1.1; using 192.168.18.43 instead (on interface wlx088af1bd9610)
26/09/30 08:48:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/30 08:48:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


# A. Extract

Membaca ketiga sumber data dari HDFS menjadi 3 SPark DataFrame

In [6]:
# Membaca data dari HDFS
df_transaksi = spark.read.csv("hdfs://localhost:9000/user/asfadani/tugas7/raw/tugas7_transaksi.csv", header=True, inferSchema=True)
df_cabang = spark.read.json("hdfs://localhost:9000/user/asfadani/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.csv("hdfs://localhost:9000/user/asfadani/tugas7/raw/tugas7_menu.csv", header=True, inferSchema=True)

df_transaksi.printSchema()
df_cabang.printSchema()
df_menu.printSchema()

root
 |-- trx_id: string (nullable = true)
 |-- cabang_id: integer (nullable = true)
 |-- menu_id: integer (nullable = true)
 |-- qty: integer (nullable = true)

root
 |-- cabang_id: long (nullable = true)
 |-- kepala_barista: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- nama_cabang: string (nullable = true)

root
 |-- menu_id: integer (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- kategori_menu: string (nullable = true)
 |-- harga: integer (nullable = true)



# Transform

**1. Transform-Join & Kolom Turunan**

Menggabungkan ke3 tabel, dan menmbahkan kolom `total_penjualan` 

In [8]:
df_gabungan = df_transaksi.join(df_menu, how="inner", on="menu_id").join(df_cabang, how="inner", on="cabang_id")

df_gabungan.show(5)

+---------+-------+------+---+---------+--------------+-----+--------------+----------+--------------------+
|cabang_id|menu_id|trx_id|qty|nama_menu| kategori_menu|harga|kepala_barista|      kota|         nama_cabang|
+---------+-------+------+---+---------+--------------+-----+--------------+----------+--------------------+
|        1|     14|   KN0|  4|  Menu-14|          Kopi|18000|         Nadia|  Magelang|Kopi Nusantara Ma...|
|        1|     11|   KN1|  4|  Menu-11|          Kopi|22000|         Nadia|  Magelang|Kopi Nusantara Ma...|
|        1|     11|   KN2|  3|  Menu-11|          Kopi|22000|         Nadia|  Magelang|Kopi Nusantara Ma...|
|        3|     11|   KN3|  1|  Menu-11|          Kopi|22000|         Nadia|  Semarang|Kopi Nusantara Se...|
|        2|     12|   KN4|  4|  Menu-12|Makanan Ringan|35000|          Reza|Yogyakarta|Kopi Nusantara Yo...|
+---------+-------+------+---+---------+--------------+-----+--------------+----------+--------------------+
only showing top 5 

In [11]:
df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))

df_gabungan.select("menu_id", "total_penjualan").show(5)

+-------+---------------+
|menu_id|total_penjualan|
+-------+---------------+
|     14|          72000|
|     11|          88000|
|     11|          66000|
|     11|          22000|
|     12|         140000|
+-------+---------------+
only showing top 5 rows



**2. Transform-Window Functions**

Menentukan top-2 menu telaris (berdasarkan `total_penjualan`) di setiap cabang, menggunankan `row_number()`

In [17]:
ringkasan_menu = df_gabungan.groupBy("nama_cabang", "menu_id", "nama_menu").agg(
    spark_sum("qty").alias("total_qty"),
    spark_sum("total_penjualan").alias("total_pendapatan")
)
ringkasan_menu.show(5)

+--------------------+-------+---------+---------+----------------+
|         nama_cabang|menu_id|nama_menu|total_qty|total_pendapatan|
+--------------------+-------+---------+---------+----------------+
|Kopi Nusantara Ma...|     12|  Menu-12|      231|         8085000|
|Kopi Nusantara Yo...|     10|  Menu-10|      202|         3636000|
|Kopi Nusantara Se...|     12|  Menu-12|      240|         8400000|
|Kopi Nusantara Se...|     14|  Menu-14|      187|         3366000|
|Kopi Nusantara Yo...|     13|  Menu-13|      181|         6335000|
+--------------------+-------+---------+---------+----------------+
only showing top 5 rows



In [18]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("nama_cabang").orderBy(col("total_qty").desc())

rank_laris = ringkasan_menu.withColumn("rank_terlaris", row_number().over(window)).filter(col("rank_terlaris") <= 2)

rank_laris.show()

+--------------------+-------+---------+---------+----------------+-------------+
|         nama_cabang|menu_id|nama_menu|total_qty|total_pendapatan|rank_terlaris|
+--------------------+-------+---------+---------+----------------+-------------+
|Kopi Nusantara Ma...|      3|   Menu-3|      260|         7800000|            1|
|Kopi Nusantara Ma...|     15|  Menu-15|      246|         7380000|            2|
|Kopi Nusantara Se...|      4|   Menu-4|      268|         5896000|            1|
|Kopi Nusantara Se...|     10|  Menu-10|      267|         4806000|            2|
|Kopi Nusantara Yo...|     15|  Menu-15|      272|         8160000|            1|
|Kopi Nusantara Yo...|     12|  Menu-12|      265|         9275000|            2|
+--------------------+-------+---------+---------+----------------+-------------+



**3. Transform-SparkSQL**

Menampilkan total penjualan per `kepala_barista` tertinggi

In [20]:
df_gabungan.createOrReplaceTempView("gabungan")

penjualan_barista = spark.sql("""
    SELECT g.kepala_barista, SUM(g.total_penjualan) AS total_penjualan
    FROM gabungan g
    GROUP BY g.kepala_barista
    ORDER BY total_penjualan DESC
""")

penjualan_barista.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



# LOAD

Simpan data gabungan (hasil bagian B) ke HDFS sebagai Parquet, path /user/[username]/tugas7/processed/laporan, dipartisi berdasarkan kategori_menu. Verifikasi dengan membaca kembali dan menghitung jumlah baris.

In [23]:
df_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet("hdfs://localhost:9000/user/asfadani/tugas7/processed/laporan")

!hdfs dfs -ls /user/asfadani/tugas7/processed/laporan

Found 4 items
-rw-r--r--   1 asfadani supergroup          0 2026-09-30 09:33 /user/asfadani/tugas7/processed/laporan/_SUCCESS
drwxr-xr-x   - asfadani supergroup          0 2026-09-30 09:33 /user/asfadani/tugas7/processed/laporan/kategori_menu=Kopi
drwxr-xr-x   - asfadani supergroup          0 2026-09-30 09:33 /user/asfadani/tugas7/processed/laporan/kategori_menu=Makanan Ringan
drwxr-xr-x   - asfadani supergroup          0 2026-09-30 09:33 /user/asfadani/tugas7/processed/laporan/kategori_menu=Non-Kopi


In [36]:
df = spark.read.parquet("hdfs://localhost:9000/user/asfadani/tugas7/processed/laporan")

print(f"Jumlah baris: {df.count()}")

Jumlah baris: 4000


# Rangkuman Eksekutif & Rekomendasi

In [30]:
df_cabang.show()

+---------+--------------+----------+--------------------+
|cabang_id|kepala_barista|      kota|         nama_cabang|
+---------+--------------+----------+--------------------+
|        1|         Nadia|  Magelang|Kopi Nusantara Ma...|
|        2|          Reza|Yogyakarta|Kopi Nusantara Yo...|
|        3|         Nadia|  Semarang|Kopi Nusantara Se...|
+---------+--------------+----------+--------------------+



In [38]:
ringkasan = df.groupBy("kota").agg(
    spark_sum("qty").alias("total_transaksi"),
    spark_sum("total_penjualan").alias("total_penjualan"),
    avg("total_penjualan").alias("avg_transaksi")
).orderBy(col("total_penjualan").desc())

ringkasan.show()

+----------+---------------+---------------+-----------------+
|      kota|total_transaksi|total_penjualan|    avg_transaksi|
+----------+---------------+---------------+-----------------+
|  Semarang|           3432|       90542000| 64997.8463747308|
|  Magelang|           3302|       87138000|66926.26728110599|
|Yogyakarta|           3205|       84645000|64862.06896551724|
+----------+---------------+---------------+-----------------+



In [41]:
ringkasan_menu.filter(col("nama_cabang") == "Kopi Nusantara Yogyakarta").orderBy(col("total_qty").asc()).show()

+--------------------+-------+---------+---------+----------------+
|         nama_cabang|menu_id|nama_menu|total_qty|total_pendapatan|
+--------------------+-------+---------+---------+----------------+
|Kopi Nusantara Yo...|     13|  Menu-13|      181|         6335000|
|Kopi Nusantara Yo...|      4|   Menu-4|      190|         4180000|
|Kopi Nusantara Yo...|      2|   Menu-2|      198|         4356000|
|Kopi Nusantara Yo...|      6|   Menu-6|      199|         4378000|
|Kopi Nusantara Yo...|     10|  Menu-10|      202|         3636000|
|Kopi Nusantara Yo...|      1|   Menu-1|      204|         7140000|
|Kopi Nusantara Yo...|     11|  Menu-11|      204|         4488000|
|Kopi Nusantara Yo...|      9|   Menu-9|      205|         7175000|
|Kopi Nusantara Yo...|      8|   Menu-8|      208|         6240000|
|Kopi Nusantara Yo...|      7|   Menu-7|      212|         3816000|
|Kopi Nusantara Yo...|      3|   Menu-3|      213|         6390000|
|Kopi Nusantara Yo...|     14|  Menu-14|      21

>Cabang dengan kinerja terbaik dipegang oleh cabang kopi yang berada di kota Semarang, yaitu Kopi Nusantara Semarang. Hal ini dikarenakan total penjualannya menjadi yang tertinggi, dengan mencatat 3432 transaksi dan total pendapatannya sebesar Rp. 90.542.000 di antara cabang kota lainnya. Tetapi, nilai rata rata penjualan tertinggi justru dipegang oleh cabang yang berada di kota Magelang. Sebaliknya, cabang di kota Jogja menjadi cabang dengan kinerja terlemah saat ini. Dari cabang Jogja tersebut, menu-13 adalah menu yang jumlah transaksinya paling sedikit. Namun, meskipun menu-13 memiliki total penjualan transaksi paling sedikit, total pendapatan yang didapat dari menu tersebut bukan yang terendah. Menu-10 menjadi menu yang total pendapatannya paling terendah. Hal tersebut disebabkan karena menu-10 memiliki harga satuan yang sangat murah. Menu (Menu-13) tersebut bisa lebih dipromosikan seperti dengan membuat paket bundling antara menu-13 dengan menu-15 yang memiliki jumlah transaksi tertinggi, sehingga transaksi menu-13 bisa lebih ditingkatkan.

# Menutup session

In [42]:
app.stopSession()

Session Tugas7 sudah dihentikan
